In [ ]:
# === auto-inserted: bev-solving src on path ===
import sys, pathlib
_root = pathlib.Path.cwd()
while _root != _root.parent and not (_root / 'src' / 'geometry.py').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# Inference for `v6_dinov2_lss_bev_cleaned` and `v7_rtmdet_cspnext_lss_bev_cleaned`

Notebook для GPU-инференса двух run'ов:
- `best.pt` и `ema_best.pt` для `v6`
- `best.pt` и `ema_best.pt` для `v7`
- подбор global threshold на `val`
- сохранение `val/test` probabilities для локального CPU-ансамбля
- опциональная сборка single-model `zip`-посылок


In [1]:
import os
import gc
import json
import math
import random
import hashlib
import zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile, ImageFilter
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

DATA_ROOT = Path('/home/jupyter/project')
DATA_TRAIN = DATA_ROOT / 'autonomy_yandex_dataset_train'
DATA_VAL = DATA_ROOT / 'autonomy_yandex_dataset_val'
DATA_TEST = DATA_ROOT / 'autonomy_yandex_dataset_test'

V6_RUN_DIR = Path('/home/jupyter/project/pray/kaggle_kernel_output/runs/v6_dinov2_lss_bev_cleaned')
V7_RUN_DIR = Path('/home/jupyter/project/runs/v7_rtmdet_cspnext_lss_bev_cleaned')
OUT_DIR = Path('/home/jupyter/project/ensemble_infer_outputs_v6_v7')
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_RTMDET_CKPT = DATA_ROOT / 'rtmdet_l_8xb32-300e_coco_20220719_112030-5a0be7c4.pth'

RUN_SPECS = [
    {'name': 'v6_dino', 'run_dir': V6_RUN_DIR},
    {'name': 'v7_rtmdet', 'run_dir': V7_RUN_DIR},
]

for p in [DATA_TRAIN, DATA_VAL, DATA_TEST, V6_RUN_DIR, V7_RUN_DIR]:
    assert p.exists(), p

print('OUT_DIR:', OUT_DIR)
print('DEFAULT_RTMDET_CKPT exists:', DEFAULT_RTMDET_CKPT.exists())


device: cuda
gpu: Tesla T4
OUT_DIR: /home/jupyter/project/ensemble_infer_outputs_v6_v7
DEFAULT_RTMDET_CKPT exists: True


In [ ]:
from src.geometry import kaggle_safe_name, load_info_with_root, remap_kaggle_paths, resolve_info_path, resolve_row_path

CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


In [3]:
def pick_existing(paths):
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError(paths)

MERGED_CLEANED_CSV = pick_existing([
    V6_RUN_DIR / 'preproc_cache' / 'merged_cleaned.csv',
    V7_RUN_DIR / 'preproc_cache' / 'merged_cleaned.csv',
    V6_RUN_DIR / 'merged_cleaned.csv',
    V7_RUN_DIR / 'merged_cleaned.csv',
])
SPLIT_NPZ = pick_existing([
    V6_RUN_DIR / 'preproc_cache' / 'test_matched_val200_split.npz',
    V7_RUN_DIR / 'preproc_cache' / 'test_matched_val200_split.npz',
    V6_RUN_DIR / 'test_matched_val200_split.npz',
    V7_RUN_DIR / 'test_matched_val200_split.npz',
])

clean_info = pd.read_csv(MERGED_CLEANED_CSV)
split = np.load(SPLIT_NPZ)
train_idx = split['train_idx'].tolist()
val_idx = split['val_idx'].tolist()

clean_info = remap_kaggle_paths(clean_info, DATA_TRAIN, DATA_VAL, DATA_TEST)
train_info = remap_kaggle_paths(clean_info.iloc[train_idx].reset_index(drop=True).copy(), DATA_TRAIN, DATA_VAL, DATA_TEST)
val_info = remap_kaggle_paths(clean_info.iloc[val_idx].reset_index(drop=True).copy(), DATA_TRAIN, DATA_VAL, DATA_TEST)
test_info = remap_kaggle_paths(load_info_with_root(DATA_TEST, 'test'), DATA_TRAIN, DATA_VAL, DATA_TEST)

print('merged_cleaned:', MERGED_CLEANED_CSV)
print('split_npz:', SPLIT_NPZ)
print('train:', len(train_info), 'val:', len(val_info), 'test:', len(test_info))

val_info.to_csv(OUT_DIR / 'val_info_export.csv', index=False)
test_info.to_csv(OUT_DIR / 'test_info_export.csv', index=False)
(DATA_TEST / 'info.csv').replace(OUT_DIR / 'test_info_official.csv') if False else None
(OUT_DIR / 'test_info_official.csv').write_bytes((DATA_TEST / 'info.csv').read_bytes())


merged_cleaned: /home/jupyter/project/pray/kaggle_kernel_output/runs/v6_dinov2_lss_bev_cleaned/preproc_cache/merged_cleaned.csv
split_npz: /home/jupyter/project/pray/kaggle_kernel_output/runs/v6_dinov2_lss_bev_cleaned/preproc_cache/test_matched_val200_split.npz
train: 4273 val: 220 test: 2000


2904011

In [ ]:
# Reusable code now lives in src/. See README.md.
from src.splits import build_rover_vocab_from_train, encode_rover


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.data import BEVDatasetV4Clean


In [ ]:
from src.models.cspnext import CSPLayer, CSPNeXtBackboneFromRTMDet, CSPNeXtBlock, ChannelAttention, ConvBNAct, SPPBottleneck, _RTMDetMultiScaleBackbone, load_rtmdet_pretrained_backbone
from src.models.strong_v4 import StrongBEVEncoderDecoder
from src.models.v7 import MultiCamBEVv7RTMDetCSPNeXtLSSClean
from src.utils.training import extract_state_dict, load_resume_state
from src.models.lss import ASPP2d, ConvGNAct, LSSViewTransform2D, ResidualBlock2d, _DINOv2MultiScaleBackbone, gn_groups


class MultiCamBEVv6DINOv2LSSClean(nn.Module):
    def __init__(self, num_rover_classes, rover_emb_dim=8, rover_cond_dim=8, n_cameras=4, freeze_backbone=False, hub_repo='facebookresearch/dinov2', backbone_name='dinov2_vitb14', backbone_out_dim=768, patch_size=14, tap_layers=(2,5,8,11), neck_dim=128, context_dim=80, depth_bins=24, depth_min=1.0, depth_max=80.0, world_z_min=-2.0, world_z_max=4.5, bev_base_channels=96, bev_gn_groups=8):
        super().__init__()
        self.n_cameras = n_cameras
        self.rover_cond_dim = rover_cond_dim
        self.backbone = _DINOv2MultiScaleBackbone(hub_repo, backbone_name, backbone_out_dim, patch_size, tap_layers, neck_dim, bev_gn_groups)
        self.view_transform = LSSViewTransform2D(neck_dim, context_dim, depth_bins, depth_min, depth_max, BEV_H, BEV_W, BEV_RES, X_RANGE, Y_RANGE, world_z_min, world_z_max, bev_gn_groups)
        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        self.rover_mlp = nn.Sequential(nn.Linear(rover_emb_dim, 16), nn.ReLU(inplace=True), nn.Linear(16, rover_cond_dim), nn.ReLU(inplace=True))
        self.bev_decoder = StrongBEVEncoderDecoder(context_dim + rover_cond_dim, bev_base_channels, bev_gn_groups)

    def forward(self, images, intrinsics, car2cams, rover_ids):
        B, N, C, H, W = images.shape
        x = images.reshape(B * N, C, H, W)
        back = self.backbone(x)
        fused = back['fused'].reshape(B, N, back['fused'].shape[1], back['fused'].shape[2], back['fused'].shape[3])
        bev = self.view_transform(fused, intrinsics, car2cams, image_hw=(H, W))
        rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
        rover_map = rover_feat.expand(-1, -1, BEV_H, BEV_W)
        logits = self.bev_decoder(torch.cat([bev, rover_map], dim=1))
        return torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)


In [ ]:
from src.utils.training import cleanup_cuda

def infer_model_type(run_name: str, ckpt: dict):
    model_type = ckpt.get('model_type')
    if model_type:
        return model_type
    if 'v6' in run_name or 'dinov2' in run_name:
        return 'v6_dinov2_lss_bev_cleaned'
    if 'v7' in run_name or 'rtmdet' in run_name:
        return 'v7_rtmdet_cspnext_lss_bev_cleaned'
    raise RuntimeError(f'Cannot infer model_type for run={run_name}')

def normalize_cfg(raw_cfg: dict, model_type: str):
    cfg = dict(raw_cfg or {})
    if model_type.startswith('v6'):
        return {
            'img_hw': tuple(cfg.get('img_hw', [384, 704])),
            'rover_emb_dim': int(cfg.get('rover_emb_dim', 8)),
            'rover_cond_dim': int(cfg.get('rover_cond_dim', 8)),
            'backbone_name': cfg.get('backbone_name', 'dinov2_vitb14'),
            'hub_repo': cfg.get('hub_repo', 'facebookresearch/dinov2'),
            'backbone_out_dim': int(cfg.get('backbone_out_dim', 768)),
            'patch_size': int(cfg.get('patch_size', 14)),
            'tap_layers': tuple(cfg.get('tap_layers', [2, 5, 8, 11])),
            'neck_dim': int(cfg.get('neck_dim', 128)),
            'context_dim': int(cfg.get('context_dim', 80)),
            'depth_bins': int(cfg.get('depth_bins', 24)),
            'depth_min': float(cfg.get('depth_min', 1.0)),
            'depth_max': float(cfg.get('depth_max', 80.0)),
            'world_z_min': float(cfg.get('world_z_min', -2.0)),
            'world_z_max': float(cfg.get('world_z_max', 4.5)),
            'bev_base_channels': int(cfg.get('bev_base_channels', 96)),
            'bev_gn_groups': int(cfg.get('bev_gn_groups', 8)),
            'use_amp': bool(cfg.get('use_amp', True)),
            'min_rover_count': int(cfg.get('min_rover_count', 30)),
            'topk_rovers': int(cfg.get('topk_rovers', 25)),
        }
    return {
        'img_hw': tuple(cfg.get('img_hw', [384, 704])),
        'rover_emb_dim': int(cfg.get('rover_emb_dim', 8)),
        'rover_cond_dim': int(cfg.get('rover_cond_dim', 8)),
        'rtmdet_backbone_ckpt': cfg.get('rtmdet_backbone_ckpt', str(DEFAULT_RTMDET_CKPT)),
        'csp_arch': cfg.get('csp_arch', 'P5'),
        'csp_deepen_factor': float(cfg.get('csp_deepen_factor', 1.0)),
        'csp_widen_factor': float(cfg.get('csp_widen_factor', 1.0)),
        'csp_expand_ratio': float(cfg.get('csp_expand_ratio', 0.5)),
        'csp_channel_attention': bool(cfg.get('csp_channel_attention', True)),
        'csp_out_indices': tuple(cfg.get('csp_out_indices', [2, 3, 4])),
        'fpn_dim': int(cfg.get('fpn_dim', 128)),
        'context_dim': int(cfg.get('context_dim', 80)),
        'depth_bins': int(cfg.get('depth_bins', 24)),
        'depth_min': float(cfg.get('depth_min', 1.0)),
        'depth_max': float(cfg.get('depth_max', 80.0)),
        'world_z_min': float(cfg.get('world_z_min', -2.0)),
        'world_z_max': float(cfg.get('world_z_max', 4.5)),
        'bev_base_channels': int(cfg.get('bev_base_channels', 96)),
        'bev_gn_groups': int(cfg.get('bev_gn_groups', 8)),
        'use_amp': bool(cfg.get('use_amp', True)),
        'min_rover_count': int(cfg.get('min_rover_count', 30)),
        'topk_rovers': int(cfg.get('topk_rovers', 25)),
    }

def build_model_for_ckpt(model_type: str, cfg: dict, rover_vocab: dict):
    if model_type.startswith('v6'):
        return MultiCamBEVv6DINOv2LSSClean(
            num_rover_classes=len(rover_vocab),
            rover_emb_dim=cfg['rover_emb_dim'],
            rover_cond_dim=cfg['rover_cond_dim'],
            freeze_backbone=False,
            hub_repo=cfg['hub_repo'],
            backbone_name=cfg['backbone_name'],
            backbone_out_dim=cfg['backbone_out_dim'],
            patch_size=cfg['patch_size'],
            tap_layers=cfg['tap_layers'],
            neck_dim=cfg['neck_dim'],
            context_dim=cfg['context_dim'],
            depth_bins=cfg['depth_bins'],
            depth_min=cfg['depth_min'],
            depth_max=cfg['depth_max'],
            world_z_min=cfg['world_z_min'],
            world_z_max=cfg['world_z_max'],
            bev_base_channels=cfg['bev_base_channels'],
            bev_gn_groups=cfg['bev_gn_groups'],
        ).to(device)
    return MultiCamBEVv7RTMDetCSPNeXtLSSClean(
        num_rover_classes=len(rover_vocab),
        rover_emb_dim=cfg['rover_emb_dim'],
        rover_cond_dim=cfg['rover_cond_dim'],
        freeze_backbone=False,
        pretrained_backbone_path=str(Path(cfg['rtmdet_backbone_ckpt']) if Path(cfg['rtmdet_backbone_ckpt']).exists() else DEFAULT_RTMDET_CKPT),
        csp_arch=cfg['csp_arch'],
        csp_deepen_factor=cfg['csp_deepen_factor'],
        csp_widen_factor=cfg['csp_widen_factor'],
        csp_expand_ratio=cfg['csp_expand_ratio'],
        csp_channel_attention=cfg['csp_channel_attention'],
        csp_out_indices=cfg['csp_out_indices'],
        fpn_dim=cfg['fpn_dim'],
        context_dim=cfg['context_dim'],
        depth_bins=cfg['depth_bins'],
        depth_min=cfg['depth_min'],
        depth_max=cfg['depth_max'],
        world_z_min=cfg['world_z_min'],
        world_z_max=cfg['world_z_max'],
        bev_base_channels=cfg['bev_base_channels'],
        bev_gn_groups=cfg['bev_gn_groups'],
    ).to(device)

def get_candidate_rover_vocab(ckpt: dict, run_cfg: dict):
    rover_vocab = ckpt.get('rover_vocab')
    if rover_vocab is not None:
        return rover_vocab
    return build_rover_vocab_from_train(train_info, min_count=run_cfg['min_rover_count'], topk=run_cfg['topk_rovers'])

def build_loaders(img_hw, rover_vocab, val_batch_size=1, test_batch_size=1):
    ds_val = BEVDatasetV4Clean(val_info, mode='val', img_hw=img_hw, rover_vocab=rover_vocab)
    ds_test = BEVDatasetV4Clean(test_info, mode='test', img_hw=img_hw, rover_vocab=rover_vocab)
    loader_val = DataLoader(ds_val, batch_size=val_batch_size, shuffle=False, num_workers=0, pin_memory=(device.type == 'cuda'))
    loader_test = DataLoader(ds_test, batch_size=test_batch_size, shuffle=False, num_workers=2, pin_memory=(device.type == 'cuda'))
    return loader_val, loader_test

def load_candidate(run_spec: dict, ckpt_kind: str):
    ckpt_path = run_spec['run_dir'] / f'{ckpt_kind}.pt'
    assert ckpt_path.exists(), ckpt_path
    ckpt = torch.load(ckpt_path, map_location='cpu')
    model_type = infer_model_type(run_spec['name'], ckpt)
    raw_cfg = ckpt.get('cfg')
    if raw_cfg is None:
        config_json = run_spec['run_dir'] / 'config.json'
        raw_cfg = json.loads(config_json.read_text()) if config_json.exists() else {}
    run_cfg = normalize_cfg(raw_cfg, model_type)
    rover_vocab = get_candidate_rover_vocab(ckpt, run_cfg)
    model = build_model_for_ckpt(model_type, run_cfg, rover_vocab)
    use_ema = (ckpt_kind == 'ema_best') and ('ema' in ckpt)
    state = ckpt['ema'] if use_ema else ckpt['model']
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"loaded {run_spec['name']}::{ckpt_kind} | model_type={model_type} | missing={len(missing)} unexpected={len(unexpected)}")
    loader_val, loader_test = build_loaders(run_cfg['img_hw'], rover_vocab)
    return {
        'run_name': run_spec['name'],
        'run_dir': run_spec['run_dir'],
        'ckpt_kind': ckpt_kind,
        'ckpt_path': ckpt_path,
        'ckpt': ckpt,
        'model_type': model_type,
        'cfg': run_cfg,
        'rover_vocab': rover_vocab,
        'model': model,
        'loader_val': loader_val,
        'loader_test': loader_test,
        'use_amp': run_cfg['use_amp'],
    }


In [8]:
@torch.inference_mode()
def collect_val_probs(model, loader, cache_path: Path):
    if cache_path.exists():
        d = torch.load(cache_path, map_location='cpu')
        return d['probs'], d['gt']
    model.eval()
    probs_list, gt_list = [], []
    for batch in tqdm(loader, desc=f'collect val probs -> {cache_path.name}'):
        images = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rover_id = batch['rover_id'].to(device)
        gt = batch['gt'].cpu()
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda' and cfg['use_amp'])):
            logits = model(images, intr, c2c, rover_id)
        probs = torch.sigmoid(logits.float()).cpu().to(torch.float16)
        probs_list.append(probs)
        gt_list.append(gt)
    probs = torch.cat(probs_list, dim=0)
    gt = torch.cat(gt_list, dim=0)
    torch.save({'probs': probs, 'gt': gt}, cache_path)
    return probs, gt


def threshold_sweep_from_cached_probs(probs, gt, thresholds):
    inter = torch.zeros(len(thresholds), dtype=torch.float64)
    union = torch.zeros(len(thresholds), dtype=torch.float64)
    valid = gt != 255
    gt_b = ((gt == 1) & valid).float()
    for i, t in enumerate(thresholds):
        pred = ((probs > t) & valid).float()
        inter[i] = (pred * gt_b).sum().item()
        union[i] = ((pred + gt_b).clamp(0, 1)).sum().item()
    return {float(t): float(inter[i] / max(union[i].item(), 1.0)) for i, t in enumerate(thresholds)}


@torch.inference_mode()
def collect_test_probs(model, loader, cache_path: Path):
    if cache_path.exists():
        return torch.load(cache_path, map_location='cpu')
    model.eval()
    probs_list = []
    for batch in tqdm(loader, desc=f'collect test probs -> {cache_path.name}'):
        images = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rover_id = batch['rover_id'].to(device)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda' and cfg['use_amp'])):
            logits = model(images, intr, c2c, rover_id)
        probs = torch.sigmoid(logits.float()).cpu().to(torch.float16)
        probs_list.append(probs)
    probs = torch.cat(probs_list, dim=0)
    torch.save(probs, cache_path)
    return probs


def candidate_slug(candidate: dict) -> str:
    return f"{candidate['run_name']}__{candidate['ckpt_kind']}"


def save_json(obj, path: Path):
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False))


def _pred_name_from_row(row):
    if 'predicted_occupancy_grid' in row:
        return Path(row['predicted_occupancy_grid']).name
    if 'predicted_static_grid' in row:
        return Path(row['predicted_static_grid']).name
    return f'{int(row.name)}.npy'

import functools
from src.submission import make_submission_from_probs as _make_submission_from_probs
def make_submission_from_probs(test_probs, threshold, out_root, tag):
    return _make_submission_from_probs(
        test_probs, threshold, tag, out_root,
        test_info=test_info, info_csv=DATA_TEST / 'info.csv',
        pred_name_fn=_pred_name_from_row,
    )


In [10]:
@torch.inference_mode()
def collect_val_probs(model, loader, cache_path: Path, use_amp: bool = True):
    if cache_path.exists():
        d = torch.load(cache_path, map_location='cpu')
        return d['probs'], d['gt']

    model.eval()
    probs_list, gt_list = [], []

    for batch in tqdm(loader, desc=f'collect val probs -> {cache_path.name}'):
        images = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rover_id = batch['rover_id'].to(device)
        gt = batch['gt'].cpu()

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda' and use_amp)):
            logits = model(images, intr, c2c, rover_id)

        probs = torch.sigmoid(logits.float()).cpu().to(torch.float16)
        probs_list.append(probs)
        gt_list.append(gt)

    probs = torch.cat(probs_list, dim=0)
    gt = torch.cat(gt_list, dim=0)
    torch.save({'probs': probs, 'gt': gt}, cache_path)
    return probs, gt


@torch.inference_mode()
def collect_test_probs(model, loader, cache_path: Path, use_amp: bool = True):
    if cache_path.exists():
        return torch.load(cache_path, map_location='cpu')

    model.eval()
    probs_list = []

    for batch in tqdm(loader, desc=f'collect test probs -> {cache_path.name}'):
        images = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rover_id = batch['rover_id'].to(device)

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda' and use_amp)):
            logits = model(images, intr, c2c, rover_id)

        probs = torch.sigmoid(logits.float()).cpu().to(torch.float16)
        probs_list.append(probs)

    probs = torch.cat(probs_list, dim=0)
    torch.save(probs, cache_path)
    return probs


In [11]:
manifest_rows = []

for run_spec in RUN_SPECS:
    for ckpt_kind in ['best', 'ema_best']:
        cleanup_cuda()
        candidate = load_candidate(run_spec, ckpt_kind)
        slug = candidate_slug(candidate)
        candidate_dir = OUT_DIR / slug
        candidate_dir.mkdir(parents=True, exist_ok=True)

        val_probs, val_gt = collect_val_probs(
            candidate['model'],
            candidate['loader_val'],
            candidate_dir / 'val_probs.pt',
            use_amp=candidate['use_amp'],
        )

        test_probs = collect_test_probs(
            candidate['model'],
            candidate['loader_test'],
            candidate_dir / 'test_probs.pt',
            use_amp=candidate['use_amp'],
        )


        meta = {
            'slug': slug,
            'run_name': candidate['run_name'],
            'ckpt_kind': candidate['ckpt_kind'],
            'model_type': candidate['model_type'],
            'ckpt_path': str(candidate['ckpt_path']),
            'img_hw': list(candidate['cfg']['img_hw']),
            'num_rover_classes': len(candidate['rover_vocab']),
            'val_probs_path': str(candidate_dir / 'val_probs.pt'),
            'test_probs_path': str(candidate_dir / 'test_probs.pt'),
            'val_shape': list(val_probs.shape),
            'test_shape': list(test_probs.shape),
        }
        save_json(meta, candidate_dir / 'meta.json')
        manifest_rows.append(meta)

        print('=' * 100)
        print(slug)
        print('val_probs:', tuple(val_probs.shape), '| test_probs:', tuple(test_probs.shape))

        del candidate['model']
        cleanup_cuda()

manifest_df = pd.DataFrame(manifest_rows).sort_values(['run_name', 'ckpt_kind']).reset_index(drop=True)
manifest_df.to_csv(OUT_DIR / 'manifest.csv', index=False)
save_json(manifest_rows, OUT_DIR / 'manifest.json')
display(manifest_df)


Using cache found in /tmp/xdg_cache/torch/hub/facebookresearch_dinov2_main


loaded v6_dino::best | model_type=v6_dinov2_lss_bev_cleaned | missing=0 unexpected=0


collect test probs -> test_probs.pt: 100%|██████████| 2000/2000 [04:22<00:00,  7.62it/s]


v6_dino__best
val_probs: (220, 1, 188, 126) | test_probs: (2000, 1, 188, 126)


Using cache found in /tmp/xdg_cache/torch/hub/facebookresearch_dinov2_main


loaded v6_dino::ema_best | model_type=v6_dinov2_lss_bev_cleaned | missing=0 unexpected=0


collect test probs -> test_probs.pt: 100%|██████████| 2000/2000 [04:26<00:00,  7.49it/s]


v6_dino__ema_best
val_probs: (220, 1, 188, 126) | test_probs: (2000, 1, 188, 126)
{
  "checkpoint": "rtmdet_l_8xb32-300e_coco_20220719_112030-5a0be7c4.pth",
  "raw_keys": 874,
  "backbone_candidate_keys": 458,
  "remapped_keys": 216,
  "loaded_keys": 458,
  "missing_keys": 0,
  "unexpected_keys": 0
}
loaded v7_rtmdet::best | model_type=v7_rtmdet_cspnext_lss_bev_cleaned | missing=0 unexpected=0


collect val probs -> val_probs.pt:   0%|          | 0/220 [00:00<?, ?it/s]


ValueError: not enough values to unpack (expected 2, got 1)

In [12]:
import shutil
import torch

# Patch v7 model so it tolerates both LSS return styles:
# - tuple: (bev, debug)
# - tensor: bev
# - dict with 'bev'
def _patched_v7_forward_debug(self, images, intrinsics, car2cams, rover_ids):
    B, N, C, H, W = images.shape
    assert N == self.n_cameras

    x = images.reshape(B * N, C, H, W)
    back = self.backbone(x)

    feat_s8 = back['feat_s8'].reshape(B, N, back['feat_s8'].shape[1], back['feat_s8'].shape[2], back['feat_s8'].shape[3])
    feat_s16 = back['feat_s16'].reshape(B, N, back['feat_s16'].shape[1], back['feat_s16'].shape[2], back['feat_s16'].shape[3])
    feat_s32 = back['feat_s32'].reshape(B, N, back['feat_s32'].shape[1], back['feat_s32'].shape[2], back['feat_s32'].shape[3])
    fused = back['fused'].reshape(B, N, back['fused'].shape[1], back['fused'].shape[2], back['fused'].shape[3])

    vt_out = self.view_transform(fused, intrinsics, car2cams, image_hw=(H, W))
    if isinstance(vt_out, tuple):
        bev, vt_debug = vt_out
    elif isinstance(vt_out, dict):
        bev = vt_out['bev'] if 'bev' in vt_out else vt_out['bev_raw']
        vt_debug = vt_out
    else:
        bev = vt_out
        vt_debug = {'bev': bev, 'depth_prob': None, 'valid_ratio': None}

    rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
    rover_map = rover_feat.expand(-1, -1, BEV_H, BEV_W)
    logits = self.bev_decoder(torch.cat([bev, rover_map], dim=1))

    return {
        'feat_s8': feat_s8,
        'feat_s16': feat_s16,
        'feat_s32': feat_s32,
        'image_fused': fused,
        'depth_prob': vt_debug.get('depth_prob', None) if isinstance(vt_debug, dict) else None,
        'bev_raw': vt_debug.get('bev', bev) if isinstance(vt_debug, dict) else bev,
        'valid_ratio': vt_debug.get('valid_ratio', None) if isinstance(vt_debug, dict) else None,
        'logits': logits,
    }

def _patched_v7_forward(self, images, intrinsics, car2cams, rover_ids):
    dbg = self.forward_debug(images, intrinsics, car2cams, rover_ids)
    return torch.nan_to_num(dbg['logits'], nan=0.0, posinf=0.0, neginf=0.0)

MultiCamBEVv7RTMDetCSPNeXtLSSClean.forward_debug = _patched_v7_forward_debug
MultiCamBEVv7RTMDetCSPNeXtLSSClean.forward = _patched_v7_forward

print("patched MultiCamBEVv7RTMDetCSPNeXtLSSClean")


patched MultiCamBEVv7RTMDetCSPNeXtLSSClean


In [13]:
# Optional: clear old failed outputs for rtmdet only
for slug in ['v7_rtmdet__best', 'v7_rtmdet__ema_best']:
    shutil.rmtree(OUT_DIR / slug, ignore_errors=True)

rtmdet_spec = next(r for r in RUN_SPECS if r['name'] == 'v7_rtmdet')
rtmdet_rows = []

for ckpt_kind in ['best', 'ema_best']:
    cleanup_cuda()

    candidate = load_candidate(rtmdet_spec, ckpt_kind)
    slug = candidate_slug(candidate)
    candidate_dir = OUT_DIR / slug
    candidate_dir.mkdir(parents=True, exist_ok=True)

    val_probs, val_gt = collect_val_probs(
        candidate['model'],
        candidate['loader_val'],
        candidate_dir / 'val_probs.pt',
        use_amp=candidate['use_amp'],
    )

    test_probs = collect_test_probs(
        candidate['model'],
        candidate['loader_test'],
        candidate_dir / 'test_probs.pt',
        use_amp=candidate['use_amp'],
    )

    meta = {
        'slug': slug,
        'run_name': candidate['run_name'],
        'ckpt_kind': candidate['ckpt_kind'],
        'model_type': candidate['model_type'],
        'ckpt_path': str(candidate['ckpt_path']),
        'img_hw': list(candidate['cfg']['img_hw']),
        'num_rover_classes': len(candidate['rover_vocab']),
        'val_probs_path': str(candidate_dir / 'val_probs.pt'),
        'test_probs_path': str(candidate_dir / 'test_probs.pt'),
        'val_shape': list(val_probs.shape),
        'test_shape': list(test_probs.shape),
    }

    save_json(meta, candidate_dir / 'meta.json')
    rtmdet_rows.append(meta)

    print('=' * 100)
    print(slug)
    print('val_probs:', tuple(val_probs.shape), '| test_probs:', tuple(test_probs.shape))

    del candidate['model']
    cleanup_cuda()

rtmdet_manifest_df = pd.DataFrame(rtmdet_rows)
rtmdet_manifest_df.to_csv(OUT_DIR / 'manifest_rtmdet_only.csv', index=False)
display(rtmdet_manifest_df)


{
  "checkpoint": "rtmdet_l_8xb32-300e_coco_20220719_112030-5a0be7c4.pth",
  "raw_keys": 874,
  "backbone_candidate_keys": 458,
  "remapped_keys": 216,
  "loaded_keys": 458,
  "missing_keys": 0,
  "unexpected_keys": 0
}
loaded v7_rtmdet::best | model_type=v7_rtmdet_cspnext_lss_bev_cleaned | missing=0 unexpected=0


collect test probs -> test_probs.pt: 100%|██████████| 2000/2000 [04:12<00:00,  7.93it/s]


v7_rtmdet__best
val_probs: (220, 1, 188, 126) | test_probs: (2000, 1, 188, 126)
{
  "checkpoint": "rtmdet_l_8xb32-300e_coco_20220719_112030-5a0be7c4.pth",
  "raw_keys": 874,
  "backbone_candidate_keys": 458,
  "remapped_keys": 216,
  "loaded_keys": 458,
  "missing_keys": 0,
  "unexpected_keys": 0
}
loaded v7_rtmdet::ema_best | model_type=v7_rtmdet_cspnext_lss_bev_cleaned | missing=0 unexpected=0


collect test probs -> test_probs.pt: 100%|██████████| 2000/2000 [04:12<00:00,  7.92it/s]


v7_rtmdet__ema_best
val_probs: (220, 1, 188, 126) | test_probs: (2000, 1, 188, 126)


,slug,run_name,ckpt_kind,model_type,ckpt_path,img_hw,num_rover_classes,val_probs_path,test_probs_path,val_shape,test_shape
0,v7_rtmdet__best,v7_rtmdet,best,v7_rtmdet_cspnext_lss_bev_cleaned,/home/jupyter/project/runs/v7_rtmdet_cspnext_l...,"[384, 704]",26,/home/jupyter/project/ensemble_infer_outputs_v...,/home/jupyter/project/ensemble_infer_outputs_v...,"[220, 1, 188, 126]","[2000, 1, 188, 126]"
1,v7_rtmdet__ema_best,v7_rtmdet,ema_best,v7_rtmdet_cspnext_lss_bev_cleaned,/home/jupyter/project/runs/v7_rtmdet_cspnext_l...,"[384, 704]",26,/home/jupyter/project/ensemble_infer_outputs_v...,/home/jupyter/project/ensemble_infer_outputs_v...,"[220, 1, 188, 126]","[2000, 1, 188, 126]"


### Что скачать на локалку после инференса

Скачай всю папку `ensemble_infer_outputs_v6_v7/`, в ней будут:
- `manifest.csv` и `manifest.json` — сводка по всем 4 кандидатам
- `val_info_export.csv` — порядок `val` объектов
- `test_info_export.csv` и `test_info_official.csv` — порядок `test` и официальный `info.csv`
- для каждого кандидата отдельная папка:
  - `val_probs.pt` — содержит `{'probs', 'gt'}`
  - `test_probs.pt`
  - `threshold_sweep.csv`
  - `meta.json`
- `single_model_submissions/` — sanity zip-посылки из лучших одиночных моделей

Для локального CPU-ансамбля тебе достаточно:
- все `val_probs.pt`
- все `test_probs.pt`
- `manifest.csv`
- `test_info_official.csv`
